# Clase 14B — RAG completo con dataset real de planes de gobierno
**Autor:** Josef Rodríguez

En esta clase construiremos una introducción seria y completa a **Retrieval Augmented Generation (RAG)** usando el dataset real del curso:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

# 1. ¿Qué problema resuelve RAG?

Los LLMs son poderosos, pero tienen tres limitaciones estructurales:

1. **Knowledge cutoff**: no necesariamente conocen documentos recientes.
2. **Hallucinations**: pueden responder con seguridad aunque la respuesta no esté sustentada.
3. **No acceden por defecto a tu base documental privada**.

RAG resuelve esto combinando:
- **recuperación de información**,
- **representaciones semánticas**,
- **generación con LLMs**.

## 1.1 Diagrama de alto nivel

```text
Pregunta del usuario
        ↓
Embedding de la pregunta
        ↓
Búsqueda en base vectorial
        ↓
Top-K fragmentos relevantes
        ↓
Construcción del prompt
        ↓
LLM
        ↓
Respuesta final
```

## Idea clave
El LLM no responde solo con "memoria interna".  
Primero se le entrega **contexto relevante recuperado**.

# 2. Fundamento conceptual: RAG no es un modelo, es una arquitectura

Esto es muy importante para tus alumnos:

- Un embedding **no es** RAG.
- FAISS **no es** RAG.
- El LLM **no es** RAG.
- RAG es la combinación orquestada de varios componentes.

## Componentes típicos
1. Corpus documental
2. Chunking
3. Embeddings
4. Vector database
5. Retriever
6. Prompt builder
7. LLM
8. Post-procesamiento / evaluación

# 3. Cargar el dataset real

Trabajaremos con los textos de planes de gobierno y los dividiremos en fragmentos para que puedan ser recuperados con mayor precisión.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"
df = pd.read_csv(url)
df = df.rename(columns={df.columns[0]: "presidente", df.columns[1]: "texto_plan"})
df["texto_plan"] = df["texto_plan"].fillna("").astype(str)

df.head()

,presidente,texto_plan
0,PABLO ALFONSO LOPEZ CHAU NAVA,formato del resumen del plan de gobierno el fo...
1,RONALD DARWIN ATENCIO SOTOMAYOR,formato del resumen del plan de gobierno el fo...
2,CESAR ACUÑA PERALTA,formato del resumen del plan de gobierno el fo...
3,JOSE DANIEL WILLIAMS ZAPATA,formato del resumen del plan de gobierno el fo...
4,ALVARO GONZALO PAZ DE LA BARRA FREIGEIRO,formato del resumen del plan de gobierno el fo...


# 4. ¿Por qué hacer chunking?

Un documento completo suele ser demasiado largo para:
- indexarlo de manera fina,
- recuperar evidencia específica,
- alimentar eficientemente a un LLM.

Por eso dividimos el documento en **chunks**.

## Definición
Chunking = proceso de dividir un documento largo en fragmentos manejables.

## Diseño
Algunas decisiones importantes:
- tamaño del chunk
- overlap
- separación por párrafos, frases o ventanas de palabras

## 4.1 Estrategia simple de chunking por palabras

Si usamos tamaño \(L\) y overlap \(o\), el chunk \(i\) puede pensarse como una ventana:

\[
chunk_i = tokens[s_i : s_i + L]
\]

donde el inicio del siguiente chunk puede avanzar en:

\[
step = L - o
\]

Esto reduce el riesgo de cortar ideas importantes.

In [2]:
def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = words[i:i+chunk_size]
        if len(chunk) == 0:
            continue
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks

# Ejemplo con el primer documento
ej_chunks = chunk_text(df["texto_plan"].iloc[0], chunk_size=120, overlap=30)
len(ej_chunks), ej_chunks[0][:500]

(170,
 'formato del resumen del plan de gobierno el formato resumen de plan de gobierno tiene como objetivo brindar al ciudadano una visión resumida del plan de gobierno presentado por las organizaciones políticas al momento de la inscripción de fórmulas y listas de candidatos. esta información será difundida en la página web de voto informado del jurado nacional de elecciones y en otros canales para conocimiento de la población. indicaciones en la primera columna, se resumirán los problemas identificad')

# 5. Crear una base de chunks

Cada chunk será una unidad recuperable dentro del sistema RAG.

In [3]:
rows = []
for _, row in df.iterrows():
    candidato = row["presidente"]
    texto = row["texto_plan"]
    chunks = chunk_text(texto, chunk_size=180, overlap=40)
    for j, ch in enumerate(chunks):
        rows.append({
            "presidente": candidato,
            "chunk_id": j,
            "chunk_text": ch
        })

chunks_df = pd.DataFrame(rows)
chunks_df.head()

,presidente,chunk_id,chunk_text
0,PABLO ALFONSO LOPEZ CHAU NAVA,0,formato del resumen del plan de gobierno el fo...
1,PABLO ALFONSO LOPEZ CHAU NAVA,1,de dichos objetivos. problemas objetivos indic...
2,PABLO ALFONSO LOPEZ CHAU NAVA,2,reciba la atención que necesita en el momento ...
3,PABLO ALFONSO LOPEZ CHAU NAVA,3,"salud y/o remodelados y oportunos, puestos en ..."
4,PABLO ALFONSO LOPEZ CHAU NAVA,4,de afectan la salud de vacunación en menores s...


In [4]:
chunks_df.shape

(881, 3)

# 6. Embeddings de chunks

Cada fragmento se convierte en un vector denso.  
Esta será la base de nuestra recuperación semántica.

In [5]:
# Si estás en un entorno limpio, descomenta:
# !pip install -q sentence-transformers faiss-cpu

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True
)

chunk_embeddings = np.array(chunk_embeddings)
chunk_embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

(881, 384)

# 7. Base vectorial con FAISS

FAISS es una librería de Meta muy usada para búsqueda de vecinos cercanos en espacios vectoriales.

## Idea matemática
Dado un embedding de consulta \(q\), queremos encontrar los \(k\) vectores \(x_i\) más cercanos según una métrica como:
- similitud coseno,
- distancia L2,
- inner product.

In [8]:
import faiss

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype("float32"))

index.ntotal

881

# 8. Retriever: recuperar los chunks más relevantes

Nuestro retriever recibirá una consulta en lenguaje natural y devolverá los chunks más útiles como evidencia.

In [9]:
def retrieve_chunks(query, top_k=5):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    out = chunks_df.iloc[indices[0]].copy()
    out["distance_l2"] = distances[0]
    return out[["presidente", "chunk_id", "chunk_text", "distance_l2"]]

retrieve_chunks("propuestas sobre educación pública y calidad educativa", top_k=5)

,presidente,chunk_id,chunk_text,distance_l2
7,PABLO ALFONSO LOPEZ CHAU NAVA,7,abandono y baja educación de implemente estrat...,0.617548
515,VLADIMIR ROY CERRON ROJAS,23,de 16. • incrementar desigual y limitado unive...,0.628153
494,VLADIMIR ROY CERRON ROJAS,2,educación pública deserción escolar (%) deserc...,0.669960
504,VLADIMIR ROY CERRON ROJAS,12,del país. universidades públicas cuenten con p...,0.678015
6,PABLO ALFONSO LOPEZ CHAU NAVA,6,"la educación superior y al trabajo, cierre bre...",0.681348


In [10]:
retrieve_chunks("hospitales, salud pública, atención primaria y telemedicina", top_k=5)

,presidente,chunk_id,chunk_text,distance_l2
166,KEIKO SOFIA FUJIMORI HIGUCHI,2,"nivel y respuesta del coordinación de propio, ...",0.611297
362,MESIAS ANTONIO GUEVARA AMASIFUEN,2,30 minutos. 4. centralización de 4. establecer...,0.638492
498,VLADIMIR ROY CERRON ROJAS,6,"• porcentaje de postergadas, al recursos en zo...",0.730720
779,ANTONIO ORTIZ VILLANO,22,13. al menos el en los servicios de en la usua...,0.741871
500,VLADIMIR ROY CERRON ROJAS,8,"comunitario, familias y resueltas en el primer...",0.787502


In [11]:
retrieve_chunks("minería, exportaciones, economía y empleo", top_k=5)

,presidente,chunk_id,chunk_text,distance_l2
298,JORGE NIETO MONTESINOS,6,avanzar en la minería ilegal. cero hectáreas p...,0.551718
717,MARIA SOLEDAD PEREZ TELLO DE RODRIGUEZ,66,y transformación años) territorio nacional min...,0.601633
273,RICARDO PABLO BELMONT CASSINELLI,16,nivel de productos centros de acopio construid...,0.632712
831,ANTONIO ORTIZ VILLANO,74,y con certificaciones proyectos sector clave p...,0.634503
215,ROBERTO HELBERT SANCHEZ PALOMINO,28,"y educación precarios, cooperativas productivi...",0.640513


# 9. Prompt construction

Una vez que recuperamos evidencia, construimos un prompt robusto.

## Plantilla recomendada

```text
Usa únicamente el contexto proporcionado para responder.
Si la respuesta no aparece en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[chunk 1]
[chunk 2]
[chunk 3]

Pregunta:
[pregunta del usuario]
```

## Principio de diseño
Un buen RAG no solo recupera bien; también **instruye bien al LLM**.

In [12]:
def build_prompt(query, retrieved_df):
    context = "\n\n".join(
        [f"[{r.presidente} | chunk {r.chunk_id}] {r.chunk_text}" for r in retrieved_df.itertuples()]
    )
    prompt = f"""Usa únicamente el contexto proporcionado para responder.
Si la respuesta no aparece en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""
    return prompt

prompt_demo = build_prompt(
    "¿Qué candidatos mencionan educación pública?",
    retrieve_chunks("educación pública", top_k=3)
)

print(prompt_demo[:2000])

Usa únicamente el contexto proporcionado para responder.
Si la respuesta no aparece en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[VLADIMIR ROY CERRON ROJAS | chunk 23] de 16. • incrementar desigual y limitado universidad pública vacantes en de manera a la educación como pilar de universidades públicas sustantiva las superior pública, justicia social y a nivel nacional vacantes en producto del desarrollo nacional, • distribución regional universidades abandono del ampliando y de vacantes en públicas, estado y de la democratizando el educación superior avanzando hacia mercantilización acceso a la pública un aumento de al neoliberal de la educación superior, • número de menos 50 % a nivel educación, que incrementando universidades públicas nacional. concentra la oferta vacantes, regionales creadas o • crear y poner universitaria en descentralizando la fortalecidas en funcionamiento pocas ciudades, oferta universitaria • presupuesto público nuevas restringe 

# 10. Evaluación conceptual del retriever

Una masterclass no termina cuando el sistema "parece funcionar".  
También debemos enseñar **cómo evaluarlo de forma rigurosa**.

En sistemas de recuperación de información (IR) y RAG, la calidad del sistema depende en gran medida del **retriever**.


## Métricas comunes en retrieval

Las métricas más utilizadas para evaluar sistemas de recuperación son:

- Precision@k
- Recall@k
- MRR (Mean Reciprocal Rank)
- nDCG (Normalized Discounted Cumulative Gain)

Estas métricas evalúan **qué tan bien el sistema recupera documentos relevantes**.


# Precision@k

Precision@k mide la proporción de documentos relevantes dentro de los **k primeros resultados recuperados**.

Sea:

- $R_k$ el conjunto de los $k$ documentos recuperados
- $Rel$ el conjunto de documentos relevantes

Entonces:

$$
Precision@k = \frac{|R_k \cap Rel|}{k}
$$

Interpretación:

- valor cercano a 1 → la mayoría de los documentos recuperados son relevantes
- valor cercano a 0 → muchos documentos irrelevantes


# Recall@k

Recall@k mide **qué proporción de todos los documentos relevantes fue recuperada** dentro de los primeros $k$ resultados.

Sea:

- $Rel$ el conjunto total de documentos relevantes

$$
Recall@k = \frac{|R_k \cap Rel|}{|Rel|}
$$

Interpretación:

- alto recall → el sistema encuentra la mayoría de documentos relevantes
- bajo recall → muchos documentos relevantes quedan fuera


# Mean Reciprocal Rank (MRR)

MRR evalúa **qué tan arriba aparece el primer documento relevante**.

Sea $rank_i$ la posición del primer documento relevante para la consulta $i$.

El **Reciprocal Rank** es:

$$
RR_i = \frac{1}{rank_i}
$$

La métrica final promedia sobre todas las consultas:

$$
MRR = \frac{1}{Q} \sum_{i=1}^{Q} \frac{1}{rank_i}
$$

donde:

- $Q$ = número de consultas evaluadas.

Interpretación:

- valor cercano a 1 → documentos relevantes aparecen muy arriba
- valor cercano a 0 → documentos relevantes aparecen muy abajo


# nDCG (Normalized Discounted Cumulative Gain)

nDCG evalúa **no solo si los documentos son relevantes, sino también su posición en el ranking**.

Primero se define:

$$
DCG@k = \sum_{i=1}^{k} \frac{2^{rel_i}-1}{\log_2(i+1)}
$$

donde:

- $rel_i$ = relevancia del documento en la posición $i$

Luego se normaliza respecto al ranking ideal:

$$
nDCG@k = \frac{DCG@k}{IDCG@k}
$$

donde $IDCG@k$ es el DCG del ranking perfecto.

Interpretación:

- $nDCG@k = 1$ → ranking ideal
- $nDCG@k \approx 0$ → ranking muy pobre


# Conexión con RAG

En sistemas **Retrieval-Augmented Generation (RAG)**:

- el **retriever** recupera documentos relevantes
- el **LLM** genera la respuesta usando esos documentos

Si el retriever falla, el LLM generará respuestas incorrectas incluso si el modelo es muy potente.

Por eso, en sistemas RAG reales, **gran parte del esfuerzo se centra en optimizar el retriever**.

# 11. Evaluación empírica del retriever

Hasta ahora hemos construido un sistema de búsqueda semántica usando embeddings.

Ahora vamos a evaluar **qué tan bien funciona el retriever**.

En sistemas reales de Retrieval-Augmented Generation (RAG), el rendimiento del sistema depende fuertemente de la calidad del retriever.

En esta sección evaluaremos el sistema usando dos métricas:

- Precision@k
- Mean Reciprocal Rank (MRR)

La idea es simple:

1. Tomamos un texto del dataset como **consulta**.
2. Buscamos los documentos más similares usando embeddings.
3. Verificamos si el documento correcto aparece entre los resultados.

Esto nos permite medir **qué tan bien el sistema recupera información relevante**.

In [17]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# -----------------------------
# 1. Cargar dataset
# -----------------------------

df = pd.read_csv("data/planes_gobierno_texto.csv")

textos = df["texto_plan"].astype(str).tolist()

print("Número de documentos:", len(textos))

# -----------------------------
# 2. Generar embeddings
# -----------------------------

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(textos, show_progress_bar=True)

# FAISS requiere float32
emb = embeddings.astype("float32")

# -----------------------------
# 3. Crear índice FAISS
# -----------------------------

dim = emb.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(emb)

print("Documentos indexados:", index.ntotal)

# -----------------------------
# 4. Evaluación del retriever
# -----------------------------

k = 5
n_queries = min(50, len(emb))

precision_scores = []
reciprocal_ranks = []

for i in range(n_queries):

    query = emb[i].reshape(1,-1)

    D, I = index.search(query, k+1)

    retrieved = I[0][1:]  # quitamos el propio documento

    relevant = i

    # Precision@k
    if relevant in retrieved:
        precision_scores.append(1/k)
    else:
        precision_scores.append(0)

    # MRR
    rank = None
    for r, doc in enumerate(retrieved, start=1):
        if doc == relevant:
            rank = r
            break

    if rank:
        reciprocal_ranks.append(1/rank)
    else:
        reciprocal_ranks.append(0)

precision_at_k = np.mean(precision_scores)
mrr = np.mean(reciprocal_ranks)

print(f"Precision@{k}: {precision_at_k:.3f}")
print("MRR:", round(mrr,3))

Número de documentos: 35


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Documentos indexados: 35
Precision@5: 0.040
MRR: 0.108


# 11. Limitaciones y decisiones de diseño

## Preguntas importantes
1. ¿Conviene chunking por palabras o por párrafos?
2. ¿Qué modelo de embedding conviene?
3. ¿Cuántos chunks recuperar?
4. ¿Conviene reranking adicional?
5. ¿Cómo evitar contexto redundante?

## Mensaje pedagógico
RAG no es solo "usar FAISS".  
RAG es una disciplina de diseño de sistemas.

# 12. Conclusiones

En esta clase aprendimos que:

1. RAG combina retrieval + prompting + generación.
2. Chunking es una decisión crítica.
3. Los embeddings permiten indexar significado, no solo palabras.
4. FAISS permite buscar rápidamente en grandes colecciones vectoriales.
5. El dataset real de planes de gobierno se presta muy bien para un sistema de preguntas y respuestas.

## Frase de cierre
**Un LLM útil para negocio no solo genera: primero busca evidencia.**